# Parsing MD Flashcards to output to PDF

## Example Text:

## Chop

Chop at ```chessboard

The last one will end at the EOF

## Latex Templates

In [206]:
old_header = r"""\documentclass[12pt]{article}
\usepackage[margin=0.5in,footskip=0.25in,letterpaper]{geometry}
\usepackage{skak}
\usepackage{chessboard}
\usepackage{graphicx}

\begin{document}"""

In [235]:
header = r"""\documentclass[12pt]{article}
\usepackage[margin=0.3in,papersize={3.0in,5.0in},footskip=0.15in]{geometry}%footskip=0.2in,
%\usepackage{skak}
\usepackage{chessboard}
\usepackage{graphicx}
\linespread{1.2}
\begin{document}"""

In [236]:
board_template = r"""{\Huge

%s

%s to move
\\
}

\scalebox{2.3}{
\chessboard[%s,
            showmover]
}

{\large \vspace{1EM} \begin{verbatim}
FEN: %s
\end{verbatim}
}
"""


In [237]:
old_q_template = r"""{\Huge
%s \\
\vspace{0.2in}

%s to move
\\
}

\newgame
\fenboard{%s}
\scalebox{3}{\showboard}

{\large \vspace{1EM} \begin{verbatim}
FEN: %s
\end{verbatim}
}
"""

In [238]:
q_template = r"""{\large
%s \\
\vspace{0.2in}

%s
\\
}

\chessboard[%s]

\pagebreak

"""



In [239]:
solution_template = r"""{\large
Solution
}

\vspace{1in}

{\large
%s
}
"""

In [240]:
old_solution_template = r"""\pagebreak

{\huge
Solution
}

\vspace{3in}

{\huge
\begin{verbatim}
%s
\end{verbatim}
}
"""

In [241]:
import re, os, copy

In [242]:
from krauss_misc import txt_mixin

In [243]:
import glob

In [244]:
ls *.md

md_flash_card_test.md


In [245]:
fn = "md_flash_card_test.md"

In [246]:
pwd

'/Users/kraussry/Work_vault_ios/chess/Analyzing_my_games/lichess_pgn_processing'

In [247]:
myfile = txt_mixin.txt_file_with_list(fn)

In [248]:
mylist = myfile.list

In [249]:
start_inds = mylist.findall("```chessboard")
start_inds

[0, 8, 16, 24]

In [250]:
end_inds = start_inds[1:]
end_inds

[8, 16, 24]

In [251]:
end_inds.append(None)

### FEN test from github

In [252]:
fen_re = re.compile(r"\s*^(((?:[rnbqkpRNBQKP1-8]+\/){7})[rnbqkpRNBQKP1-8]+)\s([b|w])\s(-|[K|Q|k|q]{1,4})\s(-|[a-h][1-8])\s(\d+\s\d+)$")

In [253]:
def fenPass(fen):
    """
    """
    #regexMatch=re.match('\s*^(((?:[rnbqkpRNBQKP1-8]+\/){7})[rnbqkpRNBQKP1-8]+)\s([b|w])\s([K|Q|k|q]{1,4})\s(-|[a-h][1-8])\s(\d+\s\d+)$', fen)
    regexMatch = fen_re.match(fen)
    if  regexMatch:
        regexList = regexMatch.groups()
        fen = regexList[0].split("/")
        if len(fen) != 8:
            raise ValueError("expected 8 rows in position part of fen: {0}".format(repr(fen)))

        for fenPart in fen:
            field_sum = 0
            previous_was_digit, previous_was_piece = False,False

            for c in fenPart:
                if c in ["1", "2", "3", "4", "5", "6", "7", "8"]:
                    if previous_was_digit:
                        raise ValueError("two subsequent digits in position part of fen: {0}".format(repr(fen)))
                    field_sum += int(c)
                    previous_was_digit = True
                    previous_was_piece = False
                elif c == "~":
                    if not previous_was_piece:
                        raise ValueError("~ not after piece in position part of fen: {0}".format(repr(fen)))
                    previous_was_digit, previous_was_piece = False,False
                elif c.lower() in ["p", "n", "b", "r", "q", "k"]:
                    field_sum += 1
                    previous_was_digit = False
                    previous_was_piece = True
                else:
                    raise ValueError("invalid character in position part of fen: {0}".format(repr(fen)))

            if field_sum != 8:
                raise ValueError("expected 8 columns per row in position part of fen: {0}".format(repr(fen)))  

    else: raise ValueError("fen doesn`t match follow this example: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1 ")  


In [254]:
class chess_chunk_parser(object):
    def __init__(self, raw_list):
        self.raw_list = txt_mixin.txt_list(raw_list)


    def parse(self):
        self.list = copy.copy(self.raw_list)
        # Assumptions: 
        # - chunk is a list of lines
        # - the first line is ```chessboard
        # - line 2 is fen: %s
        # - line 3 is orientation: %s
        # - line 4 is ```
        # - comments follow
        # - middle of comments may include '?' by itself
        #     - if it does, answer follow '?'
        line0 = self.list.pop(0)
        line0 = line0.strip()
        #print(line0)
        assert line0 == '```chessboard', "line 0 not valid: %s" % line0
        line1 = self.list.pop(0)
        line1 = line1.strip()
        assert line1.find("fen:") == 0, "problem with fen line: %s" % line1
        fen = line1[4:].strip()
        # check the fen
        fenPass(fen)
        self.fen = fen
        line2 = self.list.pop(0).strip()
        assert line2.find("orientation:") == 0, "problem with orientation line: %s" % line2
        olist = line2.split(":",1)
        myside = olist[1].strip()
        self.myside = myside
        end_md = self.list.pop(0).strip()
        assert end_md == "```", "problem with end of markdown template ```:%s." % end_md
        #what is left should be the comments:
        self.comments = copy.copy(self.list)
        self.break_comments()


    def break_comments(self):
        # self.comments should be a list
        ind = None
        for i, line in enumerate(self.comments):
            if line.strip() == '?':
                ind = i
                question = '\n'.join(mylist[0:ind])
                answer = '\n'.join(mylist[ind+1:])
                break
        if ind is None:
            self.question = '\n'.join(self.comments)
            self.answer = None
        else:
            self.question = '\n'.join(self.comments[0:ind])
            self.answer = '\n'.join(self.comments[ind+1:])

        self.question = self.question.strip()
        self.answer = self.answer.strip()


    def move_color(self):
        q = fen_re.match(self.fen)
        move_part = q.group(3)
        move_str = move_part.lower().strip()
        if 'w' in move_str:
            return "white"
        else:
            return "black"

    def build_latex(self):
        move_str = self.move_color()
        #inverse,smallboard,setfen=r1bq1rk1/p2pppbp/2p2np1/8/3BP3/2N2Q2/PPP2PPP/R3KB1R b KQ - 3 9
        opt_str = "smallboard,setfen=" + self.fen
        if self.myside == "black":
            opt_str = "inverse," + opt_str
        part1 = q_template % (self.question, move_str, opt_str)
        if self.answer is None:
            part2 = solution_template % "No answer given"
        else:
            part2 = solution_template % self.answer
        self.out = part1 + '\n' + part2
        return self.out

    

In [255]:
list_of_chunks = []

for s, e in zip(start_inds,end_inds):
    cur_chunk = mylist[s:e]
    list_of_chunks.append(cur_chunk)

In [256]:
list_of_chunks

[['```chessboard',
  'fen: r1bq1rk1/p2pppbp/2p2np1/8/3BP3/2N2Q2/PPP2PPP/R3KB1R b KQ - 3 9',
  'orientation: black',
  '```',
  'Why is Re8 bad here?',
  '?',
  'It misses a chance to play d5 and it undefends f7, allowing Bc4 to pressure mate on f7',
  ''],
 ['```chessboard',
  'fen: r1bqr1k1/p2pppbp/2p2np1/8/2BBP3/2N2Q2/PPP2PPP/R3K2R b KQ - 5 10',
  'orientation: black',
  '```',
  "The opening was mostly a success, but the database says that e5 has a 60\\% win rate for black. I didn't want to shutdown the dragon bishop.",
  '?',
  'no answer given',
  ''],
 ['```chessboard',
  'fen: r1bqr1k1/p3ppbp/2pp1np1/8/2BBP3/2N2Q2/PPP2PPP/2KR3R b - - 1 11',
  'orientation: black',
  '```',
  'I had a tunnel vision problem here. What’s wrong with c5 and what is better?',
  '?',
  'Bg4',
  ''],
 ['```chessboard',
  'fen: r1b1r1k1/p1q1ppbp/3p1np1/2B1P3/2B5/2N2Q2/PPP2PPP/2KR3R b - - 0 13',
  'orientation: black',
  '```',
  'More tunnel vision problems. There is something much better than queen take

## Parse Chunks

In [257]:
mychunks = []

for lines in list_of_chunks:
    mychunk = chess_chunk_parser(lines)
    print(mychunk)
    mychunk.parse()
    mychunks.append(mychunk)

In [258]:
print(mychunk.build_latex())

{\large
More tunnel vision problems. There is something much better than queen takes c5. \\
\vspace{0.2in}

black
\\
}

\chessboard[inverse,smallboard,setfen=r1b1r1k1/p1q1ppbp/3p1np1/2B1P3/2B5/2N2Q2/PPP2PPP/2KR3R b - - 0 13]

\pagebreak


{\large
Solution
}

\vspace{1in}

{\large
Bg4 again
}



In [259]:
myout = copy.copy(header)

first = 1
for chunk in mychunks:
    if first:
        first = 0
    else:
        myout += '\n\\pagebreak\n'
    new_str = chunk.build_latex()
    myout += new_str

myout += '\\end{document}'

In [260]:
print(myout)

\documentclass[12pt]{article}
\usepackage[margin=0.3in,papersize={3.0in,5.0in},footskip=0.15in]{geometry}%footskip=0.2in,
%\usepackage{skak}
\usepackage{chessboard}
\usepackage{graphicx}
\linespread{1.2}
\begin{document}{\large
Why is Re8 bad here? \\
\vspace{0.2in}

black
\\
}

\chessboard[inverse,smallboard,setfen=r1bq1rk1/p2pppbp/2p2np1/8/3BP3/2N2Q2/PPP2PPP/R3KB1R b KQ - 3 9]

\pagebreak


{\large
Solution
}

\vspace{1in}

{\large
It misses a chance to play d5 and it undefends f7, allowing Bc4 to pressure mate on f7
}

\pagebreak
{\large
The opening was mostly a success, but the database says that e5 has a 60\% win rate for black. I didn't want to shutdown the dragon bishop. \\
\vspace{0.2in}

black
\\
}

\chessboard[inverse,smallboard,setfen=r1bqr1k1/p2pppbp/2p2np1/8/2BBP3/2N2Q2/PPP2PPP/R3K2R b KQ - 5 10]

\pagebreak


{\large
Solution
}

\vspace{1in}

{\large
no answer given
}

\pagebreak
{\large
I had a tunnel vision problem here. What’s wrong with c5 and what is better? \\
\vspa

In [261]:
fn = 'latex_test.tex'

In [262]:
txt_mixin.dump(fn, [myout])